# RTK Wave Buoy — Publication Map Figures
Two-panel deployment site map: regional context + local satellite detail with Scripps Pier.

In [ ]:
import subprocess, sys
for pkg in ['pyubx2', 'contextily', 'pyproj', 'osmnx', 'geopandas']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('Dependencies OK')

In [ ]:
import sys
sys.path.insert(0, '../ubx_parsers')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import contextily as cx
import osmnx as ox
from pyproj import Transformer
from datetime import datetime, timezone
from v3_ubx_parser import parse_ubx_file

# suppress osmnx info logs
import logging
logging.getLogger('osmnx').setLevel(logging.WARNING)

RTK_NAMES  = {0: 'None', 1: 'Float', 2: 'Fix'}
RTK_COLORS = {0: '#aaaaaa', 1: '#f0a500', 2: '#2ca02c'}

to_3857 = Transformer.from_crs('EPSG:4326', 'EPSG:3857', always_xy=True)

## Load moored-period positions

In [ ]:
positions = parse_ubx_file('dataLog00072_moored.ubx')

records = []
for p in positions:
    try:
        dt = datetime(int(p['year']), int(p['month']), int(p['day']),
                      int(p['hour']), int(p['minute']), int(p['second']),
                      tzinfo=timezone.utc)
        records.append({
            'datetime': dt,
            'latitude': p['latitude'],
            'longitude': p['longitude'],
            'altitude_msl': p['altitude_msl'],
            'carrier_solution': p['carrier_solution'],
        })
    except Exception:
        pass

df = pd.DataFrame(records).sort_values('datetime').reset_index(drop=True)

# project to Web Mercator
df['x'], df['y'] = to_3857.transform(df['longitude'].values, df['latitude'].values)

# mooring centroid
rtk_fix = df[df['carrier_solution'] == 2]
cx_lon = rtk_fix['longitude'].mean()
cx_lat = rtk_fix['latitude'].mean()
cx_x, cx_y = to_3857.transform(cx_lon, cx_lat)

print(f"{len(df)} records  |  RTK Fix: {(df.carrier_solution==2).sum()}  Float: {(df.carrier_solution==1).sum()}  None: {(df.carrier_solution==0).sum()}")
print(f"Mooring centroid: {cx_lat:.6f}°N  {cx_lon:.6f}°E")

## Fetch Scripps Pier from OpenStreetMap

In [ ]:
pier_gdf = ox.features_from_point(
    (32.8670, -117.2540), dist=600,
    tags={'man_made': 'pier'}
).to_crs('EPSG:3857')

print(f"Pier features found: {len(pier_gdf)}")
print(pier_gdf[['geometry']].head())

## Figure — Two-panel deployment map

In [ ]:
# ── extent helpers ──────────────────────────────────────────────────────────
def latlon_buffer_3857(lat, lon, half_m):
    """Return (xmin, xmax, ymin, ymax) in EPSG:3857 for a square of 2*half_m."""
    cx_, cy_ = to_3857.transform(lon, lat)
    return cx_ - half_m, cx_ + half_m, cy_ - half_m, cy_ + half_m

# regional: 4 km half-width centred on midpoint between lab and mooring
mid_lat = (32.8683 + cx_lat) / 2
mid_lon = (- 117.2513 + cx_lon) / 2
rx0, rx1, ry0, ry1 = latlon_buffer_3857(mid_lat, mid_lon, 3000)

# local: 350 m half-width centred on mooring
lx0, lx1, ly0, ly1 = latlon_buffer_3857(cx_lat, cx_lon, 350)

# ── figure ───────────────────────────────────────────────────────────────────
fig, (ax_r, ax_l) = plt.subplots(1, 2, figsize=(13, 6))
fig.subplots_adjust(wspace=0.05)

OUTLINE = [pe.withStroke(linewidth=2.5, foreground='black')]

# ─── LEFT: regional context ──────────────────────────────────────────────────
ax_r.set_xlim(rx0, rx1)
ax_r.set_ylim(ry0, ry1)
cx.add_basemap(ax_r, crs='EPSG:3857', source=cx.providers.OpenStreetMap.Mapnik, zoom=14)

# mooring location
ax_r.plot(cx_x, cx_y, '*', color='red', ms=14, zorder=5,
          label='Mooring', markeredgecolor='white', markeredgewidth=0.8)

# SIO campus marker
sio_x, sio_y = to_3857.transform(-117.2513, 32.8683)
ax_r.plot(sio_x, sio_y, 's', color='yellow', ms=8, zorder=5,
          markeredgecolor='black', markeredgewidth=0.8, label='SIO Campus')

# box showing local-panel extent
from matplotlib.patches import Rectangle
box = Rectangle((lx0, ly0), lx1-lx0, ly1-ly0,
                edgecolor='red', facecolor='none', linewidth=1.5, zorder=6)
ax_r.add_patch(box)

# labels
ax_r.text(cx_x + 60, cx_y - 120, 'Mooring', color='white', fontsize=9,
          fontweight='bold', path_effects=OUTLINE, zorder=7)
ax_r.text(sio_x + 60, sio_y - 120, 'SIO Campus', color='white', fontsize=9,
          fontweight='bold', path_effects=OUTLINE, zorder=7)

# N arrow
ax_r.annotate('N', xy=(rx0+200, ry1-300), fontsize=11, fontweight='bold',
              ha='center', va='center', color='black',
              arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
              xytext=(rx0+200, ry1-600))

ax_r.set_axis_off()
ax_r.set_title('(a) Regional context', fontsize=11, pad=6)
ax_r.legend(loc='lower right', fontsize=8, framealpha=0.85)

# ─── RIGHT: local satellite detail ───────────────────────────────────────────
ax_l.set_xlim(lx0, lx1)
ax_l.set_ylim(ly0, ly1)
cx.add_basemap(ax_l, crs='EPSG:3857', source=cx.providers.Esri.WorldImagery, zoom=18)

# pier
if len(pier_gdf) > 0:
    pier_gdf.plot(ax=ax_l, color='cyan', linewidth=3, zorder=4, label='Scripps Pier')

# buoy track colored by RTK mode
for cs, label, color in [(0,'RTK None','#aaaaaa'), (1,'RTK Float','#f0a500'), (2,'RTK Fix','#2ca02c')]:
    sub = df[df['carrier_solution'] == cs]
    if len(sub):
        ax_l.scatter(sub['x'], sub['y'], s=5, c=color, label=label,
                     alpha=0.7, linewidths=0, zorder=5)

# scalebar — 50 m
sb_x0 = lx0 + (lx1-lx0)*0.06
sb_y0 = ly0 + (ly1-ly0)*0.06
ax_l.plot([sb_x0, sb_x0+50], [sb_y0, sb_y0], 'w-', lw=4, solid_capstyle='butt', zorder=7)
ax_l.text(sb_x0+25, sb_y0+8, '50 m', color='white', ha='center', va='bottom',
          fontsize=8, fontweight='bold', path_effects=OUTLINE, zorder=8)

ax_l.set_axis_off()
ax_l.set_title('(b) Local detail — Scripps Pier area', fontsize=11, pad=6)
ax_l.legend(loc='upper right', fontsize=8, framealpha=0.85)

fig.suptitle('RTK Wave Buoy Deployment — Scripps Institution of Oceanography',
             fontsize=12, y=1.01)

plt.savefig('fig_map.png', dpi=300, bbox_inches='tight')
print('Saved fig_map.png')
plt.show()